# Lyphocyte pseudotime endpoint DEA

In [ ]:
library_load <- suppressMessages(
    
    suppressWarnings(
    
            list(
                
                # Seurat 
                library(Seurat), 

                # Parallel 
                library(future), 

                # GSEA 
                library(msigdbr), 
                library(fgsea), 

                # Data 
                library(tidyverse), 
                library(openxlsx), 

                # Plotting 
                library(ComplexHeatmap), 
                library(circlize), 
                library(viridis), 
                library(ggplotify), 
                library(patchwork), 
                library(RColorBrewer), 

                # Python 
                library(reticulate)
            
            )
    )
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv='p.3.8.12-FD20220623HSCT', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
options(warn=-1)
options(dplyr.summarise.inform = FALSE)

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20220623HSCT")

In [ ]:
# Source files
source("plotting_global.R")
source("bin/seurat_qc.R")
source("bin/gsea.R")

In [ ]:
# Plotting Theme
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

In [ ]:
grouping_var="leiden"
random_effect_var="patient_id"

random_effect_var_cell_min=5
random_effect_var_min=3

cnt_min=3
cell_min=1
feature_group_min=3

p_adj_thr=0.1
sample_weights=TRUE

# Helper function

In [ ]:
group_select <- function(so, grouping_var, random_effect_var, random_effect_var_cell_min, random_effect_var_min) {
    
    # Get cells grouping information 
    cells_grouping <- so@meta.data[, c(grouping_var, random_effect_var)]
    
    # Count cells per group 
    cells_grouping_n <- cells_grouping %>% 
    
        # Count and filter random_effect_var by random_effect_var_cell_min
        dplyr::group_by_all() %>% 
        dplyr::summarise(random_effect_var_n=n()) %>% 
        dplyr::mutate(random_effect_var_check=ifelse(random_effect_var_n >= random_effect_var_cell_min, TRUE, FALSE)) %>% 
        dplyr::filter(random_effect_var_check) %>% 
    
        # Count and filter grouping_var by random_effect_var_min
        dplyr::group_by(across(grouping_var)) %>% 
        dplyr::mutate(grouping_var_n=sum(random_effect_var_check)) %>% 
        dplyr::mutate(grouping_var_check=ifelse(grouping_var_n >= random_effect_var_min, TRUE, FALSE)) %>% 
        dplyr::filter(grouping_var_check) %>% 

        # Ungroup 
        dplyr::ungroup() %>% as.data.frame()

    # Set cell id filter
    cells_logical <- paste(so@meta.data[[random_effect_var]], so@meta.data[[grouping_var]]) %in% paste(cells_grouping_n[[random_effect_var]], cells_grouping_n[[grouping_var]])
    
    # Set group filter 
    group_logical <- length(unique(cells_grouping_n[[grouping_var]])) > 1
    
    # Filter by cell id and return Seurat object if any cells are found
    if(any(cells_logical) & group_logical) {
        
        so <- subset(so, cells=colnames(so)[cells_logical])
        
    } else {
        
        so <- NULL
    
    }
    
    return(so)
    
}

In [ ]:
feature_select <- function(so, cnt_min, cell_min, grouping_var, random_effect_var, feature_group_min, verbose=TRUE) {
    
    # Select Seurat components 
    meta <- so@meta.data
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    
    # Prepare data for split 
    cnt <- t(as.matrix(cnt))
    cnt <- as.data.frame(cnt)
    
    # Split expression matrix by grouping variable into list
    cnt <- split(cnt, f=paste(so[[grouping_var, drop=TRUE]], so[[random_effect_var, drop=TRUE]]))
    
    # Check within group expression 
    cnt <- lapply(cnt, function(x) {colSums(x>=cnt_min)>cell_min})
    cnt <- do.call(rbind, cnt)
    
    # Check across group expression 
    cnt_check <- colSums(cnt) >= feature_group_min
    
    # Get genes 
    genes <- colnames(cnt)[cnt_check]

    # Create Seurat object with gene subset 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    cnt <- cnt[genes, , drop=FALSE]

    so <- CreateSeuratObject(counts=cnt, meta=meta)
    
    if(verbose) {message(paste("Final number of genes for testing", nrow(so)))}

    return(so)
    
}

In [ ]:
voom_lm_fit <- function(so, grouping_var, random_effect_var, sample_weights) {
    
    # Get Counts 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
        
    # Prepare count data for split 
    cnt <- t(as.matrix(cnt))
    cnt <- as.data.frame(cnt)
        
    # Make pseudobulks by suming single cell counts
    cnt <- split(cnt, f=paste0(so[[grouping_var, drop=TRUE]], "|", so[[random_effect_var, drop=TRUE]]))
    cnt <- lapply(names(cnt), function(i) {x <- data.frame(counts=colSums(cnt[[i]])); colnames(x) <- i; return(x)})
    cnt <- do.call(cbind, cnt)
        
    # Get grouping variables from cnt matrix
    grouping_var_vec <- sapply(strsplit(colnames(cnt), "\\|"), `[[`, 1) # Used for design matrix
    random_effect_var_vec <- sapply(strsplit(colnames(cnt), "\\|"), `[[`, 2) # Used as blocking variable
    
    # Design matrix 
    grouping_var_vec <- as.character(grouping_var_vec)
    design <- model.matrix(~0+grouping_var_vec)
    colnames(design) <- gsub("grouping_var_vec", "", colnames(design))
    
    suppressMessages(
        
        fit <- edgeR::voomLmFit(

            counts=cnt, 
            design=design, 
            block=random_effect_var_vec, 
            sample.weights=sample_weights, 
            var.design=design

        )
        
    )
    
    return(fit)
    
}

In [ ]:
ebayes <- function(fit) {
    
    # Get grouping var and design from fit
    grouping_var_vec <- colnames(fit$design)

    design <- model.matrix(~0+grouping_var_vec)
    colnames(design) <- gsub("grouping_var_vec", "", colnames(design))
    
    # Define result list
    result <- list()

    for(i in colnames(design)) {
        
        grouping_var_ref <- colnames(design)[!colnames(design) %in% i]

        # Contrast fit
        contrasts_vec <- paste0(i, "-(", paste0(grouping_var_ref, collapse="+"), ")/", length(grouping_var_ref))
        contrasts <- limma::makeContrasts(contrasts=contrasts_vec, levels=colnames(design))
        contrasts_fit <- limma::contrasts.fit(fit, contrasts=contrasts)

        # eBayes fit 
        efit <- limma::eBayes(contrasts_fit)

        # Get result table
        result_df <- limma::topTable(efit, sort.by="P", n=Inf, p.value=1, lfc=0, coef=1)

        # Convert results to Seurat format 
        colnames(result_df)[1] <- "avg_log2FC"
        colnames(result_df)[4] <- "p_value"
        colnames(result_df)[5] <- "p_val_adj"

        result[[i]] <- result_df

    }


    return(result)
    
}

In [ ]:
dea_vp <- function(dea, log2fc_thr=1, p_adj_thr=0.05, top_label=10, title=NULL, color_neg=RColorBrewer::brewer.pal(8, "Set1")[1], color_pos=RColorBrewer::brewer.pal(8, "Set1")[2]) {

    # Set rownames to genes
    if("gene" %in% colnames(dea)) {rownames(dea) <- dea$gene}
    
    # Annotate entries significance by log2fc_thr and p_adj_thr
    dea$p_val_adj <- ifelse(dea$p_val_adj == 0, .Machine$double.xmin, dea$p_val_adj)
    dea$sig <- ifelse(abs(dea$avg_log2FC) >= log2fc_thr & -log10(dea$p_val_adj) >= -log10(p_adj_thr), "s", "ns")
    
    # Set color based on significance and direction of dea e.g. positive and negative 
    dea$color <- ifelse(dea$sig == "s" & dea$avg_log2FC > 0, "s_pos", "ns")
    dea$color <- ifelse(dea$sig == "s" & dea$avg_log2FC < 0, "s_neg", dea$color)
    
    color <- c(color_neg, "gray", "black", color_pos)
    names(color) <- c("s_pos", "ns", "black", "s_neg")
    
    # Create labels based log2FC and p_val_adj
    dea_pos <- dea[dea$avg_log2FC > 0 & dea$sig == "s", ]
    dea_neg <- dea[dea$avg_log2FC < 0 & dea$sig == "s", ]

    pos_labels_log2FC <- dea_pos[rev(order(dea_pos$avg_log2FC)), ][1:top_label, ] %>% rownames()
    neg_labels_log2FC <- dea_neg[order(dea_neg$avg_log2FC), ][1:top_label, ] %>% rownames()
    
    pos_labels_p_val_adj <- dea_pos[order(dea_pos$p_val_adj), ][1:top_label, ] %>% rownames()
    neg_labels_p_val_adj <- dea_neg[order(dea_neg$p_val_adj), ][1:top_label, ] %>% rownames()
    
    pos_labels <- c(pos_labels_log2FC, pos_labels_p_val_adj)
    neg_labels <- c(neg_labels_log2FC, neg_labels_p_val_adj)
    
    # Set labels 
    dea$label <- ifelse(rownames(dea) %in% c(pos_labels, neg_labels), rownames(dea), NA)

    # Plot
    vp <- ggplot(dea, aes(x=avg_log2FC, y=-log10(p_val_adj), fill=dea$color, label=label), alpha=1) + 
    
        geom_point(size=4, shape=21, color="white") + 
        geom_vline(aes(xintercept=log2fc_thr), linetype="dotted", colour="black") +
        geom_vline(aes(xintercept=-log2fc_thr), linetype="dotted", colour="black") +
        geom_hline(aes(yintercept=-log10(p_adj_thr)), linetype="dotted", colour="black") +
        ggrepel::geom_text_repel(segment.color="black", force=20, force_pull=1, max.overlaps=getOption("ggrepel.max.overlaps", default=100), size=5, alpha=1, guide="none", segment.size=0.1, color="black") + 
        xlim(-max(abs(dea$avg_log2FC)), max(abs(dea$avg_log2FC))) +  
        ylim(0, max(-log10(dea$p_val_adj))+1) + 
        ggtitle(title) + xlab("average log2FC") + ylab("-log10(adj. p-value)") + 
        scale_fill_manual(values=color) + 
    
        guides(
            
            color=guide_legend(order=1, title="Group", size=2, keywidth=0.75, keyheight=0.75), 
            alpha="none"
            
        ) + 
    
        theme(

            legend.position="none", 
            aspect.ratio=1

        )
    
    return(vp)
    
}

# CD4+ T cells 

# Load data 

In [ ]:
# Store data as h5ad 
sc <- import("scanpy", as="sc", convert=FALSE)

adata <- sc$read_h5ad("data/object/pseudotime/scvi_pt_0.h5ad")

In [ ]:
# Counts
cnt <- py_to_r(adata$X)
cnt <- Matrix::t(cnt)

rownames(cnt) <- py_to_r(adata$var_names$to_list())
colnames(cnt) <- py_to_r(adata$obs_names$to_list())

# UMAP 
umap <- py_to_r(adata$obsm['X_umap'])

colnames(umap) <- c("UMAP_1", "UMAP_2")
rownames(umap) <- py_to_r(adata$obs_names$to_list())

# Meta 
meta <- py_to_r(adata$obs)

rownames(meta) <- py_to_r(adata$obs_names$to_list())
colnames(meta) <- py_to_r(adata$obs$columns$to_list())

# Store data as rds
so <- CreateSeuratObject(counts=cnt, project="lymphocyte_cd8_pt", assay="RNA")
so <- AddMetaData(so, meta)
so[["umap"]] <- CreateDimReducObject(embeddings=as.matrix(umap), key="UMAP_")

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

dplot(so, reduction="umap", group_by="leiden", alpha=1, pt_size=1.5, label=TRUE, label_size=16, label_box=TRUE)

## DEA 

In [ ]:
# Subset data
so <- subset(so, subset=leiden %in% c(0, 3, 14))
so@meta.data <- droplevels(so@meta.data)
so$leiden <- paste0("leiden_", so$leiden)

In [ ]:
# Subset features
so <- feature_select(so, cnt_min=cnt_min, cell_min=cell_min, grouping_var=grouping_var, random_effect_var=random_effect_var, feature_group_min=feature_group_min)

In [ ]:
# DEA voom fit 
fit <- voom_lm_fit(so, grouping_var=grouping_var, random_effect_var=random_effect_var, sample_weights=sample_weights) 

In [ ]:
# Get results 
results <- ebayes(fit)

In [ ]:
# Write xlsx
openxlsx::write.xlsx(results, "result/pseudotime/lymphocyte_cd4_dea.xlsx", colNames=TRUE, rowNames=TRUE)

In [ ]:
options(repr.plot.width=4*8, repr.plot.height=8)

vp <- lapply(names(results), function(i) {dea_vp(results[[i]], title=i, log2fc_thr=0, p_adj_thr=p_adj_thr)})
wrap_plots(vp, ncol=4, nrow=1)

# CD8+ T cells 

# Load data 

In [ ]:
# Store data as h5ad 
sc <- import("scanpy", as="sc", convert=FALSE)

adata <- sc$read_h5ad("data/object/pseudotime/scvi_pt_1.h5ad")

In [ ]:
# Counts
cnt <- py_to_r(adata$X)
cnt <- Matrix::t(cnt)

rownames(cnt) <- py_to_r(adata$var_names$to_list())
colnames(cnt) <- py_to_r(adata$obs_names$to_list())

# UMAP 
umap <- py_to_r(adata$obsm['X_umap'])

colnames(umap) <- c("UMAP_1", "UMAP_2")
rownames(umap) <- py_to_r(adata$obs_names$to_list())

# Meta 
meta <- py_to_r(adata$obs)

rownames(meta) <- py_to_r(adata$obs_names$to_list())
colnames(meta) <- py_to_r(adata$obs$columns$to_list())

# Store data as rds
so <- CreateSeuratObject(counts=cnt, project="lymphocyte_cd8_pt", assay="RNA")
so <- AddMetaData(so, meta)
so[["umap"]] <- CreateDimReducObject(embeddings=as.matrix(umap), key="UMAP_")

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

dplot(so, reduction="umap", group_by="leiden", alpha=1, pt_size=1.5, label=TRUE, label_size=16, label_box=TRUE)

## DEA 

In [ ]:
# Subset data
so <- subset(so, subset=leiden %in% c(1, 2, 4, 8))
so@meta.data <- droplevels(so@meta.data)
so$leiden <- paste0("leiden_", so$leiden)

In [ ]:
# Subset features
so <- feature_select(so, cnt_min=cnt_min, cell_min=cell_min, grouping_var=grouping_var, random_effect_var=random_effect_var, feature_group_min=feature_group_min)

In [ ]:
# DEA voom fit 
fit <- voom_lm_fit(so, grouping_var=grouping_var, random_effect_var=random_effect_var, sample_weights=sample_weights) 

In [ ]:
# Get results 
results <- ebayes(fit)

In [ ]:
# Write xlsx
openxlsx::write.xlsx(results, "result/pseudotime/lymphocyte_cd8_dea.xlsx", colNames=TRUE, rowNames=TRUE)

In [ ]:
options(repr.plot.width=4*8, repr.plot.height=8)

vp <- lapply(names(results), function(i) {dea_vp(results[[i]], title=i, log2fc_thr=0, p_adj_thr=p_adj_thr)})
wrap_plots(vp, ncol=4, nrow=1)